In [1]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np


def simulate_grid_delay(green_splits: np.ndarray, arrival_rates: np.ndarray, cycle_sec: float = 90.0) -> float:
    """
    Estimates average vehicle delay (in seconds) across a 3x5 signalized intersection grid.
    green_splits has shape (rows, cols) with values in [0.20, 0.80] representing green ratio.
    """
    splits = np.clip(green_splits, 0.20, 0.80)
    red_ratios = 1.0 - splits
    effective_red = red_ratios * cycle_sec

    # Queue build-up increases when arrival rate exceeds green capacity;
    # neighbour green mismatch adds platoon wait penalty.
    capacity = splits * 1.4
    congestion = np.maximum(0.0, arrival_rates - capacity)
    base_delay = 0.5 * effective_red * (1.0 + 1.8 * congestion)

    # Spatial coupling with immediate 4-way neighbours (up, down, left, right)
    up = np.roll(splits, 1, axis=0)
    down = np.roll(splits, -1, axis=0)
    left = np.roll(splits, 1, axis=1)
    right = np.roll(splits, -1, axis=1)
    sync_penalty = 2.5 * (
        np.abs(splits - up)
        + np.abs(splits - down)
        + np.abs(splits - left)
        + np.abs(splits - right)
    ) / 4.0

    total_delay = base_delay + sync_penalty
    return float(np.mean(total_delay))


def get_grid_neighbors(r: int, c: int, rows: int, cols: int):
    """Returns immediate 4-neighbour coordinates on the traffic grid."""
    neighbors = []
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        nr = (r + dr) % rows
        nc = (c + dc) % cols
        neighbors.append((nr, nc))
    return neighbors


def parallel_cellular_traffic_opt(
    arrival_rates: np.ndarray,
    rows: int = 3,
    cols: int = 5,
    n_iterations: int = 10,
    mutation_scale: float = 0.05,
    max_workers: int = 4,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)

    # Each cell holds a candidate full-grid green-split plan of shape (rows, cols)
    pop_size = rows * cols
    coords = [(r, c) for r in range(rows) for c in range(cols)]
    plans = [rng.uniform(0.25, 0.75, size=(rows, cols)) for _ in range(pop_size)]

    def eval_plan(plan):
        return simulate_grid_delay(plan, arrival_rates)

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        delays = np.array(list(pool.map(eval_plan, plans))).reshape(rows, cols)

    best_idx = int(np.argmin(delays))
    br, bc = divmod(best_idx, cols)
    best_delay = float(delays[br, bc])
    best_plan = plans[best_idx].copy()

    print("OUTPUT:")
    for it in range(1, n_iterations + 1):
        candidates = []

        for r, c in coords:
            nbrs = get_grid_neighbors(r, c, rows, cols)
            best_nr, best_nc = min(nbrs, key=lambda pos: delays[pos[0], pos[1]])

            parent_a = plans[r * cols + c]
            parent_b = plans[best_nr * cols + best_nc]

            # Blend current plan with strongest neighbouring plan + local mutation
            alpha = rng.uniform(0.3, 0.7, size=(rows, cols))
            child = alpha * parent_a + (1.0 - alpha) * parent_b
            child += rng.normal(0.0, mutation_scale, size=(rows, cols))
            child = np.clip(child, 0.20, 0.80)
            candidates.append(child)

        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            cand_delays = list(pool.map(eval_plan, candidates))

        for idx, (r, c) in enumerate(coords):
            if cand_delays[idx] <= delays[r, c]:
                plans[idx] = candidates[idx]
                delays[r, c] = cand_delays[idx]
                if cand_delays[idx] < best_delay:
                    best_delay = float(cand_delays[idx])
                    best_plan = candidates[idx].copy()

        print(f"Iteration {it}/{n_iterations} - Best Delay (s) : {best_delay:.4f}")

    return best_plan, best_delay


def main():
    rng = np.random.default_rng(42)
    rows, cols = 3, 5
    arrival_rates = rng.uniform(0.45, 1.15, size=(rows, cols))

    fixed_plan = np.full((rows, cols), 0.50)
    baseline_delay = simulate_grid_delay(fixed_plan, arrival_rates)

    best_plan, opt_delay = parallel_cellular_traffic_opt(
        arrival_rates=arrival_rates,
        rows=rows,
        cols=cols,
        n_iterations=10,
        mutation_scale=0.05,
        max_workers=4,
        seed=42,
    )

    print("\n-------- Optimization Results --------")
    print(f"Total Intersections (Grid) : {rows * cols}")
    print(f"Baseline Avg Delay (50/50) : {baseline_delay:.2f}s")
    print(f"Optimized Avg Delay        : {opt_delay:.2f}s")
    print(f"Avg Green Split            : {np.mean(best_plan):.4f}")
    print(f"Optimal Green Split Grid   : {np.round(best_plan, 2).tolist()}")


if __name__ == "__main__":
    main()

OUTPUT:
Iteration 1/10 - Best Delay (s) : 24.0128
Iteration 2/10 - Best Delay (s) : 22.7811
Iteration 3/10 - Best Delay (s) : 22.7811
Iteration 4/10 - Best Delay (s) : 22.7811
Iteration 5/10 - Best Delay (s) : 22.7811
Iteration 6/10 - Best Delay (s) : 20.6508
Iteration 7/10 - Best Delay (s) : 20.6508
Iteration 8/10 - Best Delay (s) : 20.6501
Iteration 9/10 - Best Delay (s) : 19.6002
Iteration 10/10 - Best Delay (s) : 18.5604

-------- Optimization Results --------
Total Intersections (Grid) : 15
Baseline Avg Delay (50/50) : 30.64s
Optimized Avg Delay        : 18.56s
Avg Green Split            : 0.6285
Optimal Green Split Grid   : [[0.66, 0.6, 0.71, 0.72, 0.47], [0.76, 0.6, 0.67, 0.41, 0.69], [0.59, 0.8, 0.51, 0.64, 0.59]]
